# Modeling

This notebook trains and compares multiple forecasting models on the prepared dataset.

The models include:

- classical machine learning models
- ARIMA with exogenous variables
- LSTM

The goal is to compare them using a consistent train-test split and common evaluation metrics.

In [10]:
import sys
from pathlib import Path

project_root = Path.cwd().parent
if str(project_root) not in sys.path:
    sys.path.append(str(project_root))

In [11]:
import pandas as pd

from src.data.load_data import load_dataset, split_features_target
from src.models.train_ml_models import run_ml_experiments
from src.models.train_arima import (
    train_test_split_time_series as arima_split,
    get_arima_features,
    fit_arima_model,
    evaluate_arima,
)
from src.models.train_lstm import (
    train_test_split_time_series as lstm_split,
    prepare_data,
    LSTMModel,
    train_model,
    evaluate_lstm,
)

In [12]:
df = load_dataset()
X, y = split_features_target(df)

print("Dataset shape:", df.shape)

Dataset shape: (184, 22)


## Machine Learning Models
First, I compare a set of baseline regression models using the time-series split defined in the project scripts.

In [13]:
ml_results = run_ml_experiments(X, y)
ml_results

LR: CV MSE=0.003944, Test MSE=0.001097, Direction Acc=0.432
LASSO: CV MSE=0.001160, Test MSE=0.000896, Direction Acc=0.649
EN: CV MSE=0.001160, Test MSE=0.000896, Direction Acc=0.649
KNN: CV MSE=0.001211, Test MSE=0.001022, Direction Acc=0.514
CART: CV MSE=0.002401, Test MSE=0.002984, Direction Acc=0.595
SVR: CV MSE=0.001807, Test MSE=0.002512, Direction Acc=0.351
ABR: CV MSE=0.001338, Test MSE=0.001243, Direction Acc=0.459
GBR: CV MSE=0.001564, Test MSE=0.001296, Direction Acc=0.459
RFR: CV MSE=0.001360, Test MSE=0.001348, Direction Acc=0.378
ETR: CV MSE=0.001330, Test MSE=0.001060, Direction Acc=0.486
XGB: CV MSE=0.001408, Test MSE=0.001298, Direction Acc=0.378


,model,cv_mse_mean,cv_mse_std,train_mse,test_mse,test_rmse,test_mae,directional_accuracy
0,LASSO,0.001160,0.000749,1.046235e-03,0.000896,0.029930,0.021071,0.648649
1,EN,0.001160,0.000749,1.046235e-03,0.000896,0.029930,0.021071,0.648649
2,KNN,0.001211,0.000650,8.538623e-04,0.001022,0.031963,0.023879,0.513514
3,ETR,0.001330,0.000686,2.158495e-33,0.001060,0.032565,0.024882,0.486486
4,LR,0.003944,0.003635,8.952284e-04,0.001097,0.033115,0.026606,0.432432
5,ABR,0.001338,0.000660,4.051842e-04,0.001243,0.035252,0.028785,0.459459
6,GBR,0.001564,0.000807,3.475695e-05,0.001296,0.035996,0.028387,0.459459
7,XGB,0.001408,0.000749,1.625688e-04,0.001298,0.036028,0.028634,0.378378
8,RFR,0.001360,0.000727,1.555714e-04,0.001348,0.036711,0.029381,0.378378
9,SVR,0.001807,0.001145,2.523997e-03,0.002512,0.050123,0.045674,0.351351


## ARIMA Model
Now I fit ARIMA using a smaller set of exogenous variables.
This is done to avoid making the ARIMA model too large relative to the dataset size.

In [14]:
X_train_a, X_test_a, y_train_a, y_test_a = arima_split(X, y)
X_train_arima, X_test_arima, arima_cols = get_arima_features(X_train_a, X_test_a)

print("ARIMA exogenous features used:", arima_cols)

ARIMA exogenous features used: ['GOOGL', 'IBM', 'DEXJPUS', 'DJIA', 'VIXCLS']


In [15]:
arima_model = fit_arima_model(y_train_a, X_train_arima)
arima_results, arima_predictions = evaluate_arima(
    arima_model, y_train_a, y_test_a, X_test_arima
)

arima_results

c:\Users\ASUS\Desktop\UofT\GitHub Projects\APS1052 Final Project - AI in Finance\GitHub\Financial-Time-Series-Forecasting\.venv\Lib\site-packages\statsmodels\tsa\base\tsa_model.py:473: ValueWarning: A date index has been provided, but it has no associated frequency information and so will be ignored when e.g. forecasting.
  self._init_dates(dates, freq)
c:\Users\ASUS\Desktop\UofT\GitHub Projects\APS1052 Final Project - AI in Finance\GitHub\Financial-Time-Series-Forecasting\.venv\Lib\site-packages\statsmodels\tsa\base\tsa_model.py:473: ValueWarning: A date index has been provided, but it has no associated frequency information and so will be ignored when e.g. forecasting.
  self._init_dates(dates, freq)
c:\Users\ASUS\Desktop\UofT\GitHub Projects\APS1052 Final Project - AI in Finance\GitHub\Financial-Time-Series-Forecasting\.venv\Lib\site-packages\statsmodels\tsa\base\tsa_model.py:473: ValueWarning: A date index has been provided, but it has no associated frequency information and so wil

,model,test_mse,test_rmse,test_mae,directional_accuracy,train_mse
0,ARIMA,0.000875,0.029585,0.021116,0.621622,0.001015


## LSTM Model
Finally, I train a simple LSTM as a sequence-based baseline.
Given the small dataset size, I keep the model intentionally small.

In [16]:
X_train_l, X_test_l, y_train_l, y_test_l = lstm_split(X, y)

X_train_seq, X_test_seq, y_train_seq, y_test_seq = prepare_data(
    X_train_l, X_test_l, y_train_l, y_test_l
)

lstm_model = LSTMModel(input_size=X_train_l.shape[1])
train_model(lstm_model, X_train_seq, y_train_seq, epochs=50)

Epoch 10/50, Loss: 0.001969
Epoch 20/50, Loss: 0.001833
Epoch 30/50, Loss: 0.001129
Epoch 40/50, Loss: 0.000947
Epoch 50/50, Loss: 0.000794


In [17]:
lstm_results, lstm_predictions = evaluate_lstm(
    lstm_model, X_train_seq, y_train_seq, X_test_seq, y_test_seq
)

lstm_results

,model,test_mse,test_rmse,test_mae,directional_accuracy,train_mse
0,LSTM,0.001913,0.043739,0.035041,0.5625,0.000785


## Combine Results
I now combine all model results into one comparison table.

In [18]:
final_results = pd.concat(
    [ml_results, arima_results, lstm_results],
    ignore_index=True,
    sort=False,
)

final_results = final_results.sort_values("test_mse").reset_index(drop=True)
final_results

,model,cv_mse_mean,cv_mse_std,train_mse,test_mse,test_rmse,test_mae,directional_accuracy
0,ARIMA,NaN,NaN,1.014732e-03,0.000875,0.029585,0.021116,0.621622
1,LASSO,0.001160,0.000749,1.046235e-03,0.000896,0.029930,0.021071,0.648649
2,EN,0.001160,0.000749,1.046235e-03,0.000896,0.029930,0.021071,0.648649
3,KNN,0.001211,0.000650,8.538623e-04,0.001022,0.031963,0.023879,0.513514
4,ETR,0.001330,0.000686,2.158495e-33,0.001060,0.032565,0.024882,0.486486
5,LR,0.003944,0.003635,8.952284e-04,0.001097,0.033115,0.026606,0.432432
6,ABR,0.001338,0.000660,4.051842e-04,0.001243,0.035252,0.028785,0.459459
7,GBR,0.001564,0.000807,3.475695e-05,0.001296,0.035996,0.028387,0.459459
8,XGB,0.001408,0.000749,1.625688e-04,0.001298,0.036028,0.028634,0.378378
9,RFR,0.001360,0.000727,1.555714e-04,0.001348,0.036711,0.029381,0.378378


## Modeling Takeaways

From the comparison:

- ARIMA achieved the lowest test MSE
- LASSO and Elastic Net were extremely competitive
- LSTM underperformed relative to simpler models
- this suggests that for a small financial dataset, simpler and regularized models can outperform more complex models